# CLIFFGUARD — Colab: the three things that cannot be measured on the laptop

**Runtime: T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`). Everything is sized for a free
T4: 16 GB VRAM, ~12.7 GB RAM, ~78 GB disk. Nothing here is gated — no HuggingFace token needed.

### Why this notebook is short

The full Stage 0–4 measurement is now a repo script, `scripts/run_local_ladder.py`, and it already
ran end to end on a 6 GB laptop GPU against `Qwen/Qwen2.5-1.5B-Instruct`. See
`docs/results_local_ladder.md` for those numbers.

This notebook runs the same script, unchanged, on the three things a 6 GB card and a home
connection cannot do:

| Arm | What it adds | Why not local |
|---|---|---|
| **A. Scale** | `Qwen/Qwen2.5-3B-Instruct` RTN ladder | 6.2 GB FP16 > 5.7 GB free VRAM |
| **B. Family** | `microsoft/Phi-3.5-mini-instruct` RTN ladder | 7.6 GB FP16 |
| **C. Deployment realism** | the `llama.cpp` k-quant GGUF ladder | 10 GB of downloads |

Arm A and B answer the question the laptop run cannot: **is the fitted exponent, and the collapse
bit-width `b*` it implies, a property of the quantizer or of one particular checkpoint?** A law
that only holds for one 1.5 B model is not a law.

Arm C answers a different question: does the clean RTN ladder predict what actually happens to the
k-quants people really deploy?

### One code path

Every arm shells out to the same `scripts/run_local_ladder.py`. There is no second implementation
here to drift out of sync, and each arm writes the repo's standard immutable run directory.

### Your job when it finishes

1. `File -> Download -> Download .ipynb` -> put it in `notebooks/`.
2. Unzip the results zip at the repo root so the folders land in `artifacts/runs/`.


## 0 — Environment

Installs only what Colab lacks. **`numpy` is deliberately not pinned** — forcing `numpy<2` in
Colab breaks the preinstalled torch build. `cliffguard` needs only numpy / scipy / pydantic, all
already present.


In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_BRANCH = "pivot/behavioural-rate-distortion"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        print("[drive] not mounted — results will not survive a disconnect:", exc)

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
                        REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "gguf", "bitsandbytes", "datasets"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch, numpy as np, transformers

HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "NONE"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0

print(f"repo         : {pathlib.Path.cwd()}")
print(f"python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"numpy        : {np.__version__}")
print(f"GPU          : {GPU_NAME}  ({VRAM_GB} GB VRAM)")
if hasattr(os, "statvfs"):
    st = os.statvfs(".")
    print(f"free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB  (arm C needs ~10 GB)")

if not HAS_GPU:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")
if tuple(int(p) for p in transformers.__version__.split(".")[:2]) < (4, 45):
    raise SystemExit(f"transformers {transformers.__version__} is too old for GGUF loading "
                     "(need >= 4.45). Run: !pip -q install -U transformers, then restart.")


## 1 — PREFLIGHT: does every repo API the runner calls exist and work?

Runs in seconds on CPU with synthetic data, **before** any download or GPU work. It imports every
symbol the runner uses, checks the signatures that matter, and exercises each stage function.

If this prints `PREFLIGHT OK`, no arm below can fail with an `ImportError`, `AttributeError`, or
wrong-arity `TypeError`. If it fails, stop — the notebook and the repo have drifted apart, and
running the arms would burn an hour to produce nothing.


In [ ]:
import inspect
import numpy as np

failures = []
def check(label, fn):
    try:
        fn()
        print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from cliffguard.eval.noise_floor import difference_in_means, rotation_replication, angle_between
from cliffguard.eval.isotropy import isotropy_test
from cliffguard.eval.discriminability import (
    d_prime, d_prime_with_ci, held_out_d_prime, gaussianity_gap, implied_eta,
)
from cliffguard.eval.composition import d_prime_at_bits, collapse_bits_threshold_closed_form
from cliffguard.eval.noise_spectrum import (
    EtaMeasurement, fit_eta_vs_bits_report, measure_gguf_pair, projected_perturbation_variance,
)
from cliffguard.eval.storage import new_run, record_corpus, record_environment
import scripts.run_local_ladder as runner
print("  ok    all cliffguard symbols + scripts.run_local_ladder imported")

print("signatures")
assert list(inspect.signature(measure_gguf_pair).parameters) == [
    "fp16_source", "quantized_source", "directions", "s_squared"]
print("  ok    measure_gguf_pair(fp16_source, quantized_source, directions, s_squared)")
assert "eta_by_scheme" in inspect.signature(fit_eta_vs_bits_report).parameters
print("  ok    fit_eta_vs_bits_report(eta_by_scheme: Mapping[str, EtaMeasurement])")
for name in ("rtn_quantize_dequantize", "rtn_bits_per_parameter", "collect_rtn", "main"):
    assert hasattr(runner, name), f"runner.{name} missing"
print("  ok    runner exposes rtn_quantize_dequantize / rtn_bits_per_parameter / collect_rtn")

rng = np.random.default_rng(0)
D, N = 64, 40
h0 = rng.normal(size=(N, D)) + np.eye(1, D, 0)[0] * 1.5
l0 = rng.normal(size=(N, D))
h1, l1 = h0 + rng.normal(scale=0.05, size=h0.shape), l0 + rng.normal(scale=0.05, size=l0.shape)

check("rotation_replication -> .summary()/.passes()/.z_score",
      lambda: rotation_replication(h0, l0, h1, l1, n_splits=5, seed=0).summary())
check("isotropy_test -> .is_isotropic()/.max_abs_z/.irrecoverable_fraction",
      lambda: (lambda r: (r.is_isotropic(), r.max_abs_z, r.irrecoverable_fraction))(
          isotropy_test(difference_in_means(h0, l0), difference_in_means(h1, l1),
                        n_null=20, seed=0)))
check("d_prime_with_ci -> .summary()",
      lambda: d_prime_with_ci(rng.normal(1, 1, 200), rng.normal(0, 1, 200),
                              fires_high=True, n_bootstrap=50, seed=0).summary())
check("held_out_d_prime -> (mean, std)",
      lambda: (lambda t: (float(t[0]), float(t[1])))(
          held_out_d_prime(h0, l0, n_splits=5, fires_high=True, seed=0)))
check("implied_eta / d_prime_at_bits / collapse_bits_threshold_closed_form",
      lambda: (implied_eta(1.0, 0.8), d_prime_at_bits(4.0, 2.0, 0.3),
               collapse_bits_threshold_closed_form(2.0, 0.05, 0.3)))
def _angle():
    assert abs(angle_between(h0[0], h0[0])) < 1e-6, "angle with itself is not 0"
    orthogonal = abs(angle_between(np.eye(1, D, 0)[0], np.eye(1, D, 1)[0]) - 90.0)
    assert orthogonal < 1e-6, "orthogonal vectors are not 90 degrees apart"
check("angle_between: 0 with itself, 90 for orthogonals", _angle)

def _gap():
    """Must be small for Gaussian classes and large under heavy-tail contamination."""
    clean = gaussianity_gap(rng.normal(1.5, 1, 4000), rng.normal(0, 1, 4000), fires_high=True)
    dirty = gaussianity_gap(
        np.concatenate([rng.normal(1.2, 1, 3800), rng.normal(0, 30, 200)]),
        rng.normal(0, 1, 4000), fires_high=True)
    assert clean < 0.05, f"gap {clean:.3f} too large on clean Gaussian data"
    assert dirty > 0.1, f"gap {dirty:.3f} too small on heavy-tailed data"
check("gaussianity_gap: small for gaussian, large for heavy tails", _gap)

def _projected():
    """The numerator of eta. Identical weights must give exactly zero."""
    ones = np.ones((8, 16))
    direction = rng.normal(size=8)
    assert projected_perturbation_variance(ones, ones, direction) == 0.0
    assert projected_perturbation_variance(ones, ones + rng.normal(size=(8, 16)),
                                           direction) > 0.0
check("projected_perturbation_variance: 0 for identical weights, >0 otherwise", _projected)

def _fit_roundtrip():
    m = {q: EtaMeasurement(bits_per_param_wholefile=b + 0.2, bits_per_param_payload=b,
                           eta=0.3 * 4.0 ** (4.0 - b))
         for q, b in {"a": 8.5, "b": 6.6, "c": 5.7, "d": 4.8, "e": 3.9}.items()}
    rep = fit_eta_vs_bits_report(m)
    assert abs(rep.exponent - 4.0) < 0.01, f"exponent recovery failed: {rep.exponent}"
check("fit_eta_vs_bits_report recovers a planted exponent of 4", _fit_roundtrip)

def _rtn():
    """The RTN quantizer must be exact at high precision and lossy at low."""
    w = torch.randn(64, 256, dtype=torch.float16)
    err = {b: float((runner.rtn_quantize_dequantize(w, b, 64).float() - w.float()).abs().mean())
           for b in (8, 4, 2)}
    assert err[8] < err[4] < err[2], f"RTN error not monotone in bits: {err}"
    assert runner.rtn_bits_per_parameter(4, 64) == 4.5
check("rtn_quantize_dequantize error grows as bits fall, bits/param exact", _rtn)

def _gguf_pkg():
    import gguf
    assert callable(getattr(gguf, "GGUFReader", None)) and callable(getattr(gguf, "dequantize", None))
check("gguf package exposes GGUFReader + dequantize", _gguf_pkg)

def _storage():
    r = new_run("preflight-selftest", model_id="none")
    record_environment(r)
    record_corpus(r, "x", ["a", "b"])
    r.save_array("directions", "probe", np.ones(4))
    r.save_json("probe", {"ok": True})
    r.write_manifest()
    import shutil
    shutil.rmtree(r.path)
check("storage.new_run / record_* / save_array / save_json / write_manifest", _storage)

def _sign_convention():
    """The sign error that once reported d' = -0.642. r = mean(harmful) - mean(harmless)
    makes harmful score HIGH, so every readout must use fires_high=True."""
    r = difference_in_means(h0, l0)
    r = r / np.linalg.norm(r)
    mh = (h0 @ r) / np.linalg.norm(h0, axis=1)
    ml = (l0 @ r) / np.linalg.norm(l0, axis=1)
    assert mh.mean() > ml.mean() and d_prime(mh, ml, fires_high=True) > 0
check("sign convention: fires_high=True for difference_in_means(harmful, harmless)",
      _sign_convention)

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  - " + "\n  - ".join(failures))
print("PREFLIGHT OK — every repo API these arms use exists and behaves as expected.")


## 2 — Fold A prompts

`data/` is gitignored, so the clone has no corpus. Built here with the repo's own
`scripts/download_fold_a.py --download` from `Anthropic/hh-rlhf` (MIT), the same code path the
laptop run used, and cached to Drive so a reconnect does not re-download it.

"Refused" means the dataset's *rejected* response looks like a refusal — a property of the corpus,
not of the model under test. That is a ceiling on d' and is recorded in every manifest.


In [ ]:
FOLD_A = pathlib.Path("data/folds/fold_a")
NEEDED = ["anthropic_hh_refused.jsonl", "anthropic_hh_benign.jsonl"]
DRIVE_FOLD = DRIVE_ROOT / "fold_a"

def have_corpus():
    return all((FOLD_A / f).exists() for f in NEEDED)

if not have_corpus() and DRIVE_FOLD.exists():
    import shutil
    FOLD_A.mkdir(parents=True, exist_ok=True)
    for f in DRIVE_FOLD.glob("*.jsonl"):
        shutil.copy(f, FOLD_A / f.name)
    print("[prompts] restored from Drive")

if not have_corpus():
    print("[prompts] building Fold A ...")
    # No --max: the script's verify() checks against the blueprint minima (1200/800/600) and
    # returns 1 for anything smaller, so a cap would make a good corpus look like a failure.
    # The exit code is informational; the real check is have_corpus() plus the runner's own
    # unique-prompt count.
    proc = subprocess.run([sys.executable, "scripts/download_fold_a.py", "--download"],
                          capture_output=True, text=True)
    print(proc.stdout[-2500:])
    if not have_corpus():
        print(proc.stderr[-2000:])
        raise SystemExit("Fold A build failed. Refusing to substitute synthetic prompts.")
    if DRIVE_ROOT.exists():
        import shutil
        DRIVE_FOLD.mkdir(parents=True, exist_ok=True)
        for f in FOLD_A.glob("*.jsonl"):
            shutil.copy(f, DRIVE_FOLD / f.name)
        print("[prompts] cached to Drive")

for f in NEEDED:
    n = sum(1 for ln in (FOLD_A / f).read_text(encoding="utf-8").splitlines() if ln.strip())
    print(f"  {f}: {n} lines")


## 3 — The driver

Every arm is one call to `scripts/run_local_ladder.py`. Output is streamed so a stall is visible,
and the tail of each log is kept for the summary at the end.

`N_PER_CLASS = 250` is the pre-registered minimum-detectable-effect sample size (80 % power at the
9.46 deg population rotation). Lowering it makes a null result uninterpretable — change it only if
you hit a wall-clock limit, and say so when you report.


In [ ]:
N_PER_CLASS = 250
SEED = 0
RESULTS = {}

def run_arm(label, args, timeout=7200):
    """Stream one runner invocation; keep its tail and exit status."""
    cmd = [sys.executable, "scripts/run_local_ladder.py",
           "--n", str(N_PER_CLASS), "--seed", str(SEED), "--label", label] + args
    print("$", " ".join(cmd), flush=True)
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            if "Loading weights" in line or "it/s]" in line:
                continue                      # progress bars, not results
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait(timeout=timeout)
    except Exception as exc:
        proc.kill()
        lines.append(f"ABORTED: {type(exc).__name__}: {exc}")
    RESULTS[label] = {"returncode": proc.returncode, "minutes": (time.time() - started) / 60,
                      "tail": lines[-40:]}
    status = "OK" if proc.returncode == 0 else f"FAILED (rc={proc.returncode})"
    print(f"\n=== {label}: {status} in {RESULTS[label]['minutes']:.1f} min ===\n", flush=True)
    return proc.returncode == 0

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


## 4 — ARM A: does the law survive a change of scale?

`Qwen/Qwen2.5-3B-Instruct`, 36 layers, read at layer 18 (mid-depth, matching the 1.5 B run's
14/28). Same RTN ladder, same group size, same corpus.

**What would refute generality:** a fitted exponent whose interval does not overlap the 1.5 B one,
or a collapse threshold `b*` that moves by more than a bit. **What would support it:** overlapping
exponents and a `b*` that shifts smoothly with `d'_0` — which is what Theorem 2 predicts, since
`b*` depends on the model only through `d'_0` and `eta_4`.


In [ ]:
free_vram()
run_arm("colab-rtn-qwen3b",
        ["--model", "Qwen/Qwen2.5-3B-Instruct", "--layer", "18",
         "--ladder-kind", "rtn", "--bits", "8", "7", "6", "5", "4", "3", "2",
         "--group", "64", "--splits", "50"])


## 5 — ARM B: does it survive a change of family?

`microsoft/Phi-3.5-mini-instruct`, 32 layers, read at layer 16. A different tokenizer, a different
pretraining corpus, a different safety-tuning recipe, and fused QKV/gate-up projections.

Two models from one lab is a coincidence; two labs is the beginning of a law. If Phi's exponent
also brackets 4 and its `b*` follows the same `d'_0` relation, assumption A3 is a property of
round-to-nearest quantization rather than of Qwen.

`--skip-nf4` because the NF4 arm is a categorical comparison that adds nothing to the ordinal
bit-width axis, and skipping it saves a model load.


In [ ]:
free_vram()
run_arm("colab-rtn-phi35",
        ["--model", "microsoft/Phi-3.5-mini-instruct", "--layer", "16",
         "--ladder-kind", "rtn", "--bits", "8", "7", "6", "5", "4", "3", "2",
         "--group", "64", "--splits", "50", "--skip-nf4"])


## 6 — ARM C: does the clean ladder predict the deployed one?

`Qwen/Qwen2.5-1.5B-Instruct-GGUF` — the real `llama.cpp` k-quant ladder (F16, Q8_0, Q6_K, Q5_K_M,
Q4_K_M, Q3_K_M, Q2_K), all converted by Qwen from one F16 checkpoint. About 10 GB of downloads,
which is why this is a Colab arm and not a laptop one.

This is the honest counterweight to the RTN ladder. RTN varies **only** bit-width, which is what
makes it a clean ordinal dose axis — but nobody deploys RTN. k-quants vary block structure and
per-tensor type assignment as well, so they are a messier axis and a more realistic one. The
comparison worth reporting is whether `b*` fitted on RTN lands anywhere near the k-quant curve.

Each rung is dequantized into a dense fp16 torch model, so peak VRAM is one model copy, not the sum
of the ladder. `gguf.dequantize` was verified locally to handle every k-quant type here at the real
`ffn_down` shape.


In [ ]:
free_vram()
run_arm("colab-gguf-qwen1.5b",
        ["--model", "Qwen/Qwen2.5-1.5B-Instruct",
         "--gguf-repo", "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
         "--gguf-stem", "qwen2.5-1.5b-instruct",
         "--layer", "14", "--ladder-kind", "gguf", "--splits", "50"])


In [ ]:
# Clears the UNVERIFIED-AGAINST-REAL-FILE marker on scripts/verify_gguf_pair.py.
# Tensor-name and shape correspondence between F16 and Q4_K_M is also the precondition
# arm C's eta measurement depends on, so this is a gate rather than a formality.
from huggingface_hub import hf_hub_download
try:
    f16 = hf_hub_download("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-fp16.gguf")
    q4 = hf_hub_download("Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q4_k_m.gguf")
    proc = subprocess.run([sys.executable, "scripts/verify_gguf_pair.py", f16, q4],
                          capture_output=True, text=True)
    print(proc.stdout[-4000:])
    RESULTS["verify_gguf_pair"] = {"returncode": proc.returncode, "tail": proc.stdout[-2000:]}
    if proc.returncode == 0:
        print("\nverify_gguf_pair PASSED on a real F16/Q4_K_M pair — report this so the")
        print("UNVERIFIED-AGAINST-REAL-FILE marker can be removed from the script header.")
    else:
        print(proc.stderr[-2000:])
        print("\nFAILED — treat arm C's eta numbers as unverified until this passes.")
except Exception as exc:
    print("skipped:", type(exc).__name__, exc)


## 7 — OPTIONAL: the AWQ / GPTQ arm (the actual test of C5)

**Run this only after section 8 has downloaded your results.** Installing `autoawq` or `gptqmodel`
can replace Colab's torch build and kill the runtime, so it is placed last: failure then costs
nothing already earned.

Both repos are public and derived from the same base checkpoint as arm C's F16:
`Qwen/Qwen2.5-1.5B-Instruct-AWQ` and `Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4`.

C5 predicts these are the schemes whose perturbation is **not** isotropic, because AWQ and GPTQ
protect salient channels while RTN and k-quants do not. If they are isotropic too, **C5 is refuted**
and the literature split must come from judges or corpora instead — equally worth reporting.

This is a *categorical* comparison. AWQ and GPTQ must never be placed on the ordinal bit-width axis
alongside the RTN rungs: they are different algorithms, not different doses.


In [ ]:
ENABLE_AWQ_GPTQ = False   # flip to True only after your results zip is downloaded

if not ENABLE_AWQ_GPTQ:
    print("Skipped. Set ENABLE_AWQ_GPTQ = True and rerun this cell to add the arm.")
else:
    print("Installing AWQ/GPTQ backends. If the runtime dies, arms A-C are already saved.\n")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "autoawq"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gptqmodel"], check=False)
    print("torch still importable:", torch.__version__, "| cuda:", torch.cuda.is_available())

    import numpy as np
    from cliffguard.eval.noise_floor import difference_in_means, rotation_replication
    from cliffguard.eval.isotropy import isotropy_test
    from cliffguard.eval.discriminability import held_out_d_prime
    import scripts.run_local_ladder as runner

    runner.MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    harmful, benign = runner.load_prompts(N_PER_CLASS)
    cache = pathlib.Path("artifacts/colab_awq_cache/Qwen-Qwen2.5-1.5B-Instruct")
    cache.mkdir(parents=True, exist_ok=True)
    fp16_h, fp16_l = runner.collect("FP16", runner.load_fp16_model, harmful, benign, 14, cache)
    r_fp16 = difference_in_means(fp16_h, fp16_l)

    def loader_for(repo):
        def _load():
            from transformers import AutoModelForCausalLM
            try:
                return AutoModelForCausalLM.from_pretrained(repo, dtype=torch.float16,
                                                            device_map={"": 0})
            except TypeError:
                return AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16,
                                                            device_map={"": 0})
        return _load

    stage1b = {}
    for tag, repo in {"AWQ_INT4": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
                      "GPTQ_INT4": "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"}.items():
        try:
            h, low = runner.collect(tag, loader_for(repo), harmful, benign, 14, cache)
            direction = difference_in_means(h, low)
            rep = rotation_replication(fp16_h, fp16_l, h, low, n_splits=50, seed=SEED)
            iso = isotropy_test(r_fp16, direction, n_null=400, seed=SEED)
            dp_mean, dp_std = held_out_d_prime(h, low, n_splits=50, fires_high=True, seed=SEED)
            stage1b[tag] = {"replicates": rep.passes(), "replication_z": rep.z_score,
                            "angle_deg": iso.angle_deg, "max_abs_z": iso.max_abs_z,
                            "concentration_null_not_rejected": iso.is_isotropic(),
                            "irrecoverable_fraction": iso.irrecoverable_fraction,
                            "d_prime_held_out": dp_mean, "d_prime_held_out_std": dp_std}
            print(f"\n{tag}: {rep.summary()}\n{tag}: {iso.summary()}"
                  f"\n{tag}: held-out d' = {dp_mean:+.4f} +/- {dp_std:.4f}")
        except Exception as exc:
            stage1b[tag] = {"error": f"{type(exc).__name__}: {exc}"}
            print(f"\n{tag}: UNAVAILABLE — {type(exc).__name__}: {exc}")
            print("     Report this as a SKIPPED arm; infer nothing from its absence.")
        finally:
            free_vram()

    from cliffguard.eval.storage import new_run, record_corpus, record_environment
    run = new_run("colab-awq-gptq", model_id=runner.MODEL_ID,
                  extra={"note": "categorical comparison; NOT on the ordinal bit-width axis",
                         "gpu": GPU_NAME, "n_per_class": N_PER_CLASS, "layer": 14})
    record_environment(run)
    record_corpus(run, "harmful_refused", harmful)
    record_corpus(run, "benign", benign)
    run.save_json("stage1b_awq_gptq", stage1b)
    run.write_manifest()
    run.append_to_index("AWQ/GPTQ categorical isotropy comparison")
    RESULTS["colab-awq-gptq"] = {"returncode": 0, "tail": [json.dumps(stage1b, indent=2)]}
    print(f"\nrun: {run.path}")


## 8 — Collect and download

Zips every run directory this session produced, mirrors to Drive, and downloads one file.


In [ ]:
import shutil

runs = sorted(pathlib.Path("artifacts/runs").glob("*colab-*"))
print("run directories produced this session:")
for r in runs:
    print(f"  {r.name}")
if not runs:
    raise SystemExit("No colab run directories found — every arm failed. Check the logs above.")

stamp = time.strftime("%Y%m%d-%H%M%S")
staging = pathlib.Path(f"/content/cliffguard_colab_{stamp}") if IN_COLAB \
    else pathlib.Path(f"cliffguard_colab_{stamp}")
staging.mkdir(parents=True, exist_ok=True)
for r in runs:
    shutil.copytree(r, staging / "artifacts" / "runs" / r.name, dirs_exist_ok=True)
(staging / "arm_status.json").write_text(json.dumps(RESULTS, indent=2), encoding="utf-8")

zip_path = pathlib.Path(shutil.make_archive(str(staging), "zip",
                                            root_dir=staging.parent, base_dir=staging.name))
print(f"\n{zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")

if DRIVE_ROOT.exists():
    shutil.copy(zip_path, DRIVE_ROOT / zip_path.name)
    print(f"mirrored to {DRIVE_ROOT / zip_path.name}")

print("\narm status:")
for label, info in RESULTS.items():
    rc = info.get("returncode")
    print(f"  {label:24s} {'OK' if rc == 0 else f'FAILED rc={rc}'}"
          + (f"  {info['minutes']:.1f} min" if "minutes" in info else ""))

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print("auto-download failed — take it from the Files pane or Drive:", exc)

print(f"""
DONE. Two things to put in the repo:

  1. File -> Download -> Download .ipynb  ->  notebooks/colab_ladder_and_eta_EXECUTED.ipynb
  2. unzip {zip_path.name}, then move its artifacts/runs/* into the repo's artifacts/runs/

Then say "colab results are in".
""")


---
## Reading the outcome

Decide what each row means **before** looking at the numbers.

| Result | Meaning |
|---|---|
| 3 B and Phi exponent intervals overlap the 1.5 B one | **A3 generalises.** The decay base is a property of RTN quantization, not of one checkpoint. |
| They do not overlap | **A3 is model-dependent.** Theorem 2 still holds per-model with a measured base; the universal claim does not. |
| `b*` tracks `d'_0` across all three models | **C3 supported** — this is the headline: a collapse point predicted from weights and a full-precision score, with no low-precision evaluation. |
| Out-of-sample d' lands within 1 sd on every model | **C2/C3 supported.** This is the paper. |
| eta ratio spread far from 1 | **F5 fires — C2's mechanism is refuted** even if the d' decay itself is real. The most important negative outcome. |
| k-quant curve far from the RTN prediction | The clean ladder does not transfer to deployed quantizers. A real limitation, and publishable. |
| AWQ/GPTQ isotropy null rejected, k-quants not | **C5 supported** — salience-aware quantizers perturb anisotropically. |
| AWQ/GPTQ also fail to reject | **C5 refuted.** The literature split comes from judges or corpora. Still publishable. |
| Rotation stops replicating at the lowest rungs | Not a failure — at some bit-width the direction is *destroyed* rather than rotated, and the geometric claims stop applying below it. Report the bit-width where it happens. |

### Known limits, stated up front

- **No completions are generated**, so nothing here supports a *behavioural* claim. Margin
  discriminability is a proxy for behaviour, not behaviour. Real generations plus a
  StrongREJECT-style judge is the next stage and is not built.
- **Labels are corpus labels.** "Refused" describes the hh-rlhf rejected response, not what these
  models do. That caps d' independently of quantization.
- **eta is a proxy.** Summing per-matrix projected variances assumes independent layers and
  isotropic unit-variance activations; neither holds. The scale-free ratio test is the form that
  survives this.
- **One layer, one language, one corpus** per model.
